# Step 17.4 — Adam Optimizer

## 从参数梯度树，到有历史的参数更新规则

17.3说明了如何从scalar loss得到与可训练参数对应的梯度树。本节按现有roadmap继续讲 **first moment（一阶矩）**、**second moment（二阶矩）**、**optimizer state（优化器状态）**，以及这些状态为什么占用内存。

材料审核方已独立核验17.3真实母版、文档、MLX v0.32.2接口和完整教学回复，核心静态PASS，实际修正 **0/2**。这是材料验收，不是学员掌握或运行反馈。17.4本次初始生成实际修正计数为 **0/2**。

本节准备一个已有梯度输入下的Adam接口例子，不组装forward、loss、求导与更新的完整训练步骤。**本轮止于17.4，不进入17.5。** 所有代码单元均未执行、无输出；没有运行Notebook、测试、训练、实验、采样或用户work副本，没有读取/下载语料，也不代答练习。


## 0. 新 Work Copy：本节需要的内容全部自带

沿用项目 Python 3.12 / `.venv` kernel 与固定依赖 `mlx==0.32.2`。导入：

- `mlx.core as mx`：创建明确的float32数组；
- `mlx.optimizers as optim`：使用MLX的优化器模块；`as optim` 只是给模块取一个较短的Python别名。

本节自带参数树、梯度树、Adam构造与状态示例。之后使用新副本时，从本节imports开始按顺序使用即可；无需旧kernel变量、旧Notebook的 `%run`、语料、sampler或完整MiniGPT定义，也不增加项目依赖。

我们会把17.3已经手算出的梯度写成一个显式输入，专门研究优化器如何处理它。这不代表本轮运行了 `nn.value_and_grad`，也不替代真实模型所需的重新求导。


In [ ]:
import mlx.core as mx
import mlx.optimizers as optim

# Imports for a future work copy. This master remains unexecuted.


## 1. Gradient 与 Optimizer：输入是什么，新增的职责是什么？

17.3的小模型只有一片参数叶子 `head → bias`，shape为 `(3,)`。在三个初始分数均为0、四个已展平targets为 `[2,1,0,2]` 时，手算得到：

$$
L=\ln3,\qquad
\nabla_\theta L=\left[\frac1{12},\frac1{12},-\frac16\right].
$$

这里的L已经是四个token-level losses的mean。这个梯度已经包含求平均的影响，交给优化器时**不再除以B×T=4**。

**Optimizer（优化器）** 使用当前参数、当前梯度以及自己保存的状态，计算下一组参数。标量loss说明当前表现，梯度说明各参数的局部敏感度；Adam的参数更新接口接收梯度树，不是只接收一个loss数值就能推断所有参数怎样改变。

本节使用以下数学记号：

| 记号 | 含义 |
|---|---|
| \(\theta_{k-1}\) | 第k次更新前的参数 |
| \(g_k\) | 在这组参数处得到的loss梯度；本节用显式手算值作为输入 |
| \(k=1,2,\ldots\) | 优化器更新的编号，不是token位置t、序列长度T或监督项数量B×T |
| \(\theta_k\) | 应用第k次更新规则后得到的参数 |

实际模型有很多数组，代码继续按参数树处理；数学向量只为了简洁地表达逐分量规则。

### 自包含的参数树与示例梯度树

Python的 `{"head": {"bias": ...}}` 是嵌套 **dict（字典）**。字符串是key；`parameters["head"]["bias"]` 沿两层key读取数组叶子。参数树和梯度树的对应路径、叶子shape必须匹配。本例直接写出普通字典树，这是MLX优化器接受的接口；无需为了研究状态而再次创建Module。

`mx.array(..., dtype=mx.float32)` 把列表变为32位浮点数组。下面 `1.0 / 12.0` 等Python除法写出手算分数的浮点表示；在机器上不承诺分数能被精确表示。变量名 `example_grads` 特别标明它是示例输入，未由本轮自动微分产生。


In [ ]:
# EXPLICIT INTERFACE EXAMPLE — prepared only, NOT executed.

parameters = {
    "head": {
        "bias": mx.array([0.0, 0.0, 0.0], dtype=mx.float32),
    }
}

example_grads = {
    "head": {
        "bias": mx.array(
            [1.0 / 12.0, 1.0 / 12.0, -1.0 / 6.0],
            dtype=mx.float32,
        ),
    }
}

# Both leaves have shape (3,).
# These are the explicit zero parameters and hand-derived gradient from 17.3.
# No loss/gradient function is called, and the gradient is not divided by 4 again.


## 2. 一阶矩m、二阶原始矩v：每个参数分量都有自己的历史

只看当前梯度，得到的是当前位置的一次信息。Adam还保存两份滚动摘要：一份记梯度方向与大小的平滑历史，另一份记梯度平方的平滑历史。这里的“矩”来自统计量：一阶关注数值本身，二阶原始矩关注数值的平方；不是对loss再求一次导数。

**Exponential moving average，EMA（指数移动平均）** 每次保留旧摘要的一部分，再加入新信息的一部分。系数 \(\beta_1,\beta_2\) 满足 \(0\leq\beta<1\)；越接近1，旧历史保留得越多。初始化 \(m_0=v_0=0\)，然后：

$$
m_k=\beta_1m_{k-1}+(1-\beta_1)g_k,
$$

$$
v_k=\beta_2v_{k-1}+(1-\beta_2)g_k^2.
$$

- **First moment estimate（一阶矩估计）m**：梯度的指数加权平均，保留符号；方向相反的历史贡献可以相互抵消。
- **Second raw moment estimate（二阶原始矩估计）v**：梯度平方的指数加权平均，记录大小尺度；平方不会因正负号相反而抵消。

为什么叫“指数”平均？以m为例，第二次更新可以展开为：

$$
m_2=\beta_1(1-\beta_1)g_1+(1-\beta_1)g_2.
$$

旧梯度每经过一次更新，就再乘一个 \(\beta_1\)，所以更旧的贡献带着更高次幂。代码只需要保存上一次摘要m，不必保存完整梯度历史。

上述平方、乘法、加法以及后面的开方、除法都 **element-wise（逐元素）** 进行。例如 `[a,b,c]` 的平方是 `[a²,b²,c²]`，不是点积，也不是矩阵乘法。因此参数、梯度、m、v的相应叶子shape始终一致；本例全部是 `(3,)`。

v没有减去均值平方，不能称为通常意义上的 **variance（方差）**；它也不是二阶导数或参数间的协方差矩阵。所谓Adam使用“一阶与二阶矩”，仍然只需要17.3提供的一阶梯度。


## 3. 更新公式、Bias Correction与MLX的实际默认值

先补齐更新公式中的两个数：

- \(\eta\)（eta）是 **learning rate（学习率）**，控制整体更新尺度；
- \(\epsilon\)（epsilon）是一个很小的正数，本例取 \(10^{-8}\)，加在分母中改善数值稳定性。

不开启偏差修正时，本版本Adam使用：

$$
\theta_k=\theta_{k-1}-\eta\frac{m_k}{\sqrt{v_k}+\epsilon}.
$$

\(\sqrt{v_k}\) 是逐元素非负平方根，它把梯度平方的尺度变回与梯度相同的尺度。m保留平滑后的方向，分母按各分量的历史大小调节更新尺度；在m和其他条件固定时，较大的v会减小该分量的更新幅度。减号表示沿所构造方向的反方向调整，但这种历史更新规则不保证每次更新后loss都下降。

### 为什么会有Bias Correction？

m、v从0开始，早期历史权重之和不足1。用一个纯数学例子看：如果每次梯度始终等于同一个g，那么

$$
m_1=(1-\beta_1)g,\qquad
m_2=(1-\beta_1^2)g,\qquad
m_k=(1-\beta_1^k)g.
$$

它还没有达到g，原因包含初始0留下的影响。v同理。**Bias correction（偏差修正）** 除掉这部分零初始化权重因子：

$$
\hat m_k=\frac{m_k}{1-\beta_1^k},\qquad
\hat v_k=\frac{v_k}{1-\beta_2^k}.
$$

帽子 \(\hat{}\) 标记“修正后的估计”，\(\beta^k\) 表示β的k次方。开启修正后使用：

$$
\theta_k=\theta_{k-1}-\eta\frac{\hat m_k}{\sqrt{\hat v_k}+\epsilon}.
$$

修正针对零初始化带来的估计偏差，不是保证准确估计变化中的所有梯度，也不是保证loss下降。这里的bias与参数路径 `head.bias` 中的 **bias parameter（偏置参数）** 含义不同。

注意两种写法的epsilon都在平方根**外面**，不能改成 \(\sqrt{v+\epsilon}\)。本版本True分支与上面的带帽公式在实数代数上对应；浮点计算顺序与舍入不要求逐位一致。

### 固定MLX v0.32.2：默认值与本节选择分开记录

已核对 [v0.32.2优化器源码](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/optimizers/optimizers.py)。在线官方文档可能随版本更新；本课以固定版本为准。

| 构造参数 | MLX v0.32.2默认行为 | 本节显式设置 |
|---|---|---|
| `learning_rate` | 必填，没有默认值 | `1e-3`，即0.001 |
| `betas` | `[0.9, 0.999]` | `[0.9, 0.999]` |
| `eps` | `1e-8` | `1e-8` |
| `bias_correction` | **`False`** | **`True`**，对应带帽公式 |

`1e-3` 与 `1e-8` 是Python科学计数法；`betas` 中两项依次对应β₁和β₂。`True`/`False` 是布尔值。使用 `name=value` 这样的 **keyword argument（关键字实参）**，可以清楚记录每项设置。

本节为了连接经典带偏差修正公式，主动选择 `bias_correction=True`。它不是MLX默认值；本例的学习率也不是MLX提供的默认值或对真实MiniGPT的调参结论。构造Adam只创建优化器对象，不求梯度，也不改变参数。


In [ ]:
# EXPLICIT LESSON CONFIGURATION — prepared only, NOT executed.

optimizer = optim.Adam(
    learning_rate=1e-3,
    betas=[0.9, 0.999],
    eps=1e-8,
    bias_correction=True,  # Deliberate lesson choice; MLX 0.32.2 defaults to False.
)

# At construction, global state has step=0 and learning_rate.
# Per-parameter m/v state is introduced by initialization below.


### 一个小手算：沿用17.3的三参数梯度

本例从 \(\theta_0=[0,0,0]\)、\(m_0=v_0=[0,0,0]\) 开始；第1次更新输入：

$$
g_1=\left[\frac1{12},\frac1{12},-\frac16\right].
$$

保留实际默认的 \(\beta_1=0.9\)、\(\beta_2=0.999\)，所以新梯度的权重分别是0.1与0.001：

$$
m_1=0.1g_1=\left[\frac1{120},\frac1{120},-\frac1{60}\right],
$$

$$
v_1=0.001g_1^2
=\left[\frac1{144000},\frac1{144000},\frac1{36000}\right].
$$

第三个梯度是负数，m的第三项仍为负；v的第三项为正。由于本节显式开启修正，第一步有：

$$
\hat m_1=\frac{m_1}{0.1}=g_1,\qquad
\hat v_1=\frac{v_1}{0.001}=g_1^2.
$$

因此精确的实数公式为：

$$
\theta_1=-0.001\,
\frac{[1/12,\ 1/12,\ -1/6]}
{[1/12,\ 1/12,\ 1/6]+10^{-8}},
$$

这里向量除法和标量加法都逐元素进行。暂时忽略相对这些分量极小的epsilon，仅为了看更新尺度，就得到：

$$
\theta_1\approx[-0.001,-0.001,+0.001].
$$

这只是这组非零梯度在首步开启修正时的近似；不能推出Adam每次每个分量总更新一个learning rate。

**如果使用MLX默认的False会怎样？** m₁、v₁仍是上面相同的数，但不做除以0.1与0.001的修正。第一个分量在暂时忽略epsilon时：

$$
\frac{m_{1,0}}{\sqrt{v_{1,0}}}
=\frac{1/120}{1/(120\sqrt{10})}=\sqrt{10}.
$$

于是同一学习率下，参数约为 `[-0.0031623, -0.0031623, +0.0031623]`。这个比较只用于说明设置会改变首步尺度，不增加第二个运行例子，也不把True或False当作调参胜负结论。

全部数值都是按给定输入做的手算与近似，未执行求导、优化器更新或数值验证。实际代码保留 `eps=1e-8`；分数与小数会经过float32舍入。


## 4. Optimizer State：参数路径下各保存m与v

**State（状态）** 是优化器在两次调用之间保留的信息。参数只告诉我们模型当前的位置；m、v记录到达当前位置时的梯度历史，step记录已经应用了多少次优化器更新。

### 先初始化，才能按路径查看m/v

固定版本构造Adam时，`optimizer.state` 已包含全局 `step` 与 `learning_rate`；此时还没有为每个参数建立m/v。`optimizer.state` 是属性，直接读取，不写成 `optimizer.state()`。

`optimizer.init(parameters)` 是 **method（方法）** 调用：把参数树交给优化器，按每片叶子建立状态。它不返回新参数，不执行学习更新，也不增加step。显式初始化是可选的；如果尚未初始化，首次 `apply_gradients` 会从传入的梯度树自动初始化。本例显式传参数树，便于先看清状态。

在v0.32.2中，Adam用 `mx.zeros_like(parameter)` 初始化每片叶子的m和v；`zeros_like` 表示生成与给定数组相同shape、相同dtype的全零数组。因此本例参数是float32，初始m/v也是float32。不要由此声称“MLX Adam状态不论输入类型都固定为float32”。

接下来沿 `optimizer.state["head"]["bias"]` 进入该参数的状态字典，再用key `"m"` 或 `"v"` 取到相应数组。`.shape` 查看数组形状，`.dtype` 查看元素类型。


In [ ]:
# STATE INITIALIZATION — prepared only, NOT executed.

optimizer.init(parameters)

print("parameter shape:", parameters["head"]["bias"].shape)
print("gradient shape:", example_grads["head"]["bias"].shape)
print("m shape:", optimizer.state["head"]["bias"]["m"].shape)
print("v shape:", optimizer.state["head"]["bias"]["v"].shape)
print("m dtype:", optimizer.state["head"]["bias"]["m"].dtype)
print("v dtype:", optimizer.state["head"]["bias"]["v"].dtype)

# By construction: all four leaf shapes are (3,), all four dtypes are float32.
# Initialization keeps step at 0 and m/v at zero; no parameter update here.


### 实际状态路径与含义

下面是结构说明，不是Notebook运行输出：

| 路径 | Shape / 含义 |
|---|---|
| `parameters["head"]["bias"]` | `(3,)`，当前模型参数 |
| `example_grads["head"]["bias"]` | `(3,)`，这次提供的梯度 |
| `optimizer.state["head"]["bias"]["m"]` | `(3,)`，该参数叶子的未修正一阶矩 |
| `optimizer.state["head"]["bias"]["v"]` | `(3,)`，该参数叶子的未修正二阶原始矩 |
| `optimizer.state["step"]` | `()`，uint64计数，初始化时为0 |
| `optimizer.state["learning_rate"]` | `()`，本例固定学习率数组 |

不是在顶层各放一棵 `state["m"]` 与 `state["v"]` 树，而是沿参数路径进入后，再读取该叶子的m/v。参数有更多层或数组时，仍按对应路径组织。开启bias correction也继续保存原始递推得到的m、v；\(\hat m,\hat v\) 是更新公式里的修正量，不是额外两份永久历史。

`betas`、`eps`、`bias_correction` 是优化器对象的配置属性，不在这个 `state` 字典内。因此state不能自动代表全部配置；这里仅说明状态含义，保存/恢复实现留在现有后续课纲。


### 一次孤立的Adam接口调用：谁被改变，返回什么？

固定接口为 `optimizer.apply_gradients(gradients, parameters)`。实参顺序是梯度树在前、参数树在后。对于本例匹配的树，它会：

1. 使用已有初始化状态，将全局step从0推进到1；
2. 用输入梯度更新每个参数路径下的m/v；
3. 用本节选定的修正公式计算新参数，返回对应的新参数树。

赋值 `parameters_after = ...` 把返回树接到新变量。此Adam实现不会把新参数直接写入本例原来的 `parameters` 字典；优化器自身的state则会变化，所以它不是一个没有状态变化的纯函数。

一次调用会处理整棵梯度树，step不按参数分量或B×T个监督位置分别递增。本例第1次更新使用k=1进行偏差修正，不用k=0去除以零。

下面只准备这个已有梯度输入下的状态转移接口。没有MiniGPT forward、loss调用、自动微分调用或训练循环，也不加入17.5的求值边界。


In [ ]:
# ONE ISOLATED OPTIMIZER TRANSITION — prepared only, NOT executed.

parameters_after = optimizer.apply_gradients(example_grads, parameters)

m_after = optimizer.state["head"]["bias"]["m"]
v_after = optimizer.state["head"]["bias"]["v"]

print("returned parameter shape:", parameters_after["head"]["bias"].shape)
print("stored m shape:", m_after.shape)
print("stored v shape:", v_after.shape)

# Hand-derived first-call expectations, not observed outputs:
# step = 1
# m = [1/120, 1/120, -1/60]
# v = [1/144000, 1/144000, 1/36000]
# parameters_after is approximately [-0.001, -0.001, +0.001] at head.bias.
# The approximation ignores the small epsilon; the actual API keeps eps=1e-8.


### 保留历史意味着什么？

后续参数更新需要继续使用同一个优化器所维护的m、v、step；每次都重新构造Adam，会丢掉已积累的历史。新参数、旧参数、当前梯度和优化器状态承担不同职责，不能只保留一个数组就认为信息完整。

另一方面，本节的固定 `example_grads` 只代表初始零参数处的那一次手算梯度。真实后续更新需要对新的参数与相应数据取得新的梯度；不能反复套用它就当作MiniGPT训练。

如果以后在work副本里重复调用上面的 `apply_gradients` 单元，同一个optimizer的状态会继续前进，结果不能再按“首次调用”的手算解释。要回到本例起点，应重新建立本节参数/示例梯度、重新构造Adam，再初始化后使用一次示例。仅再次调用 `init` 不是承诺重置所有现有历史的接口。

这里说明的是未来使用副本时的状态语义，本轮没有进行调用、重复运行或恢复实验。


## 5. 为什么Adam State占内存？

设所有参与优化的参数共有 **P个标量元素**，每个元素占d字节。P不是参数数组的个数，也不是B×T个token targets的数量。例如一个shape为 `(2,3)` 的参数数组含6个标量；本例一片 `(3,)` 叶子只有P=3。

每个参数分量要保留一份m与一份v。若参数与矩状态采用相同dtype，仅这两份逐参数状态的数据量就是：

$$
\text{moment-state bytes}=2Pd.
$$

这解释了 **persistent state（跨更新保留的状态）** 的开销：保存平滑历史需要额外数组。更新次数增加时m/v的shape不变；它们覆盖上一份摘要，而不是把所有历史梯度不断追加进去。这个说法讨论状态数组规模，不承诺运行时图、缓存或进程内存恒定。

本节所有相关数组采用float32，即每元素4字节：

| 数组数据 | 一般P | 本例P=3 |
|---|---:|---:|
| 参数θ | 4P字节 | 12字节 |
| 当前梯度g，如果同时保存 | 4P字节 | 12字节 |
| 一阶矩m | 4P字节 | 12字节 |
| 二阶矩v | 4P字节 | 12字节 |
| **仅m/v两份状态合计** | **8P字节** | **24字节** |

如果同时把θ、g、m、v四类数组都算入，本例逻辑数据共48字节。这个数不包括全局step/learning_rate等小状态、Python容器、中间激活、临时数组、运行时与缓存，也不计算本教学例保留新旧参数等额外对象。**48字节不是进程实际占用或峰值测量。**

例如仅作同float32条件下的数量级估算，P=1,000,000时，参数数据为4MB，m/v额外为8MB；这里MB按1,000,000字节计。不要把“两份额外参数规模的矩状态”误写成“训练总内存恒为参数的三倍或四倍”。

### 把数量级与数组metadata对应

下面使用数组的 **metadata（元信息）**：`.size` 表示元素个数，`.itemsize` 表示每元素字节数，`.nbytes` 表示该数组逻辑数据的字节数。它们是属性，读取时不加括号。这里只看本例的一片叶子；真实多叶参数树需要把各叶子的相应数据量相加。

这段准备代码用于读懂数据规模，不调用进程内存监测或建立内存实验。类型与元信息接口也按固定v0.32.2源码核对。


In [ ]:
# LOGICAL ARRAY PAYLOAD — prepared only, NOT executed; not a memory benchmark.

parameter_leaf = parameters["head"]["bias"]
gradient_leaf = example_grads["head"]["bias"]
m_leaf = optimizer.state["head"]["bias"]["m"]
v_leaf = optimizer.state["head"]["bias"]["v"]

P = parameter_leaf.size
bytes_per_element = parameter_leaf.itemsize
moment_payload_bytes = m_leaf.nbytes + v_leaf.nbytes
four_array_payload_bytes = (
    parameter_leaf.nbytes
    + gradient_leaf.nbytes
    + moment_payload_bytes
)

print("parameter scalars:", P)
print("bytes per parameter element:", bytes_per_element)
print("m/v logical payload bytes:", moment_payload_bytes)
print("theta/g/m/v logical payload bytes:", four_array_payload_bytes)

# By the declared shapes/dtypes: P=3, d=4, m/v=24 bytes, four arrays=48 bytes.
# Excludes optimizer scalars, containers, extra arrays, activations and runtime.
# These are derived counts, not observed process-memory measurements.


## 本节的Shape、使用顺序与停止点

| 对象 | 本例Shape | 角色 |
|---|---|---|
| 17.3的scalar loss | `()` | 上节已由四项位置loss等权平均得到；本节不重算 |
| 当前参数与输入梯度叶子 | `(3,)` / `(3,)` | 相同参数路径下的值与局部导数 |
| m与v叶子 | `(3,)` / `(3,)` | 优化器保留的两份逐参数历史摘要 |
| 返回的新参数叶子 | `(3,)` | 应用一次选定Adam规则后的参数 |
| step与learning_rate | `()` / `()` | 全局更新计数与本例固定学习率 |

之后使用新work副本时：

1. 在仓库根目录 `git pull`，用时间戳新文件名与 `cp -n` 把75母版复制到已有 `notebooks/work/`，只打开副本。
2. 选择项目Python3.12 / `.venv` kernel，从本节imports与显式树开始；没有旧课运行依赖。
3. 阅读m/v、偏差修正和固定版本默认值，再使用显式Adam构造代码。
4. 阅读手算与状态路径，再使用初始化与一次孤立状态转移示例；保留这次的True设置与首步前提。
5. 根据shape与dtype理解状态规模，再阅读元信息代码和逻辑字节数。

这些是未来副本的可操作顺序，本轮只准备材料。没有执行Notebook、测试、训练、实验、采样或用户work副本，没有读取/下载语料，也未代答练习；材料静态审阅和发布不等于学员掌握或运行验证。

**本轮只准备并发布17.4，完成即停，不进入17.5。** 不返工已合格章节。累计实际修正：16.1=**1/2**、16.2=**1/2**，16.3至16.5及17.1至17.3各**0/2**，17.4初版=**0/2**。每小节最多两次实际修正，跨轮与换对话不重置；后续按请求逐小节至大Step20全部完成后停止。


## 参考与课程衔接

- [17.3母版：参数树与nn.value_and_grad](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/decce21f9de3490833b0ab7023579bf0718a18a2/notebooks/74_step17_3_nn_value_and_grad.ipynb)：scalar loss、三参数手算梯度与返回树。
- [MLX v0.32.2优化器实现](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/optimizers/optimizers.py)：本节默认值、两种修正设置、状态结构、init与apply_gradients的依据。
- [MLX v0.32.2数组接口实现](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/array.cpp)：size、itemsize与nbytes属性。
- [Official Adam API](https://ml-explore.github.io/mlx/build/html/python/optimizers/_autosummary/mlx.optimizers.Adam.html)、[Optimizer.init](https://ml-explore.github.io/mlx/build/html/python/optimizers/_autosummary/mlx.optimizers.Optimizer.init.html)、[Optimizer.apply_gradients](https://ml-explore.github.io/mlx/build/html/python/optimizers/_autosummary/mlx.optimizers.Optimizer.apply_gradients.html)：公开接口说明；在线页面可能更新，课程固定源码为最终版本依据。
- [Kingma and Ba, Adam: A Method for Stochastic Optimization](https://arxiv.org/abs/1412.6980)：算法中的矩估计与零初始化偏差修正；本节不展开论文证明或其他优化器。
- [Course roadmap](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/main/docs/course-roadmap.md) 与 [teaching handoff](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/main/docs/teaching-handoff.md)：本节范围、材料状态和累计计数。
